# BAN 612 — Week 4 In-Class Notebook
## API Data Collection: Requests, JSON, DataFrames, and Validation

### What you will practice

1. Sending a request to an API
2. Passing parameters with `params=`
3. Checking status codes and final URLs
4. Inspecting JSON before converting it to a DataFrame
5. Converting API responses into tables
6. Cleaning data types after an API request
7. Logging issues rather than silently skipping failures
8. Using an API for a simple market-analysis practice exercise

### Important principle

An API returns structured data. It does **not** automatically return trustworthy, analysis-ready data.

## 1. Import libraries

We use:

- `requests` to send HTTP requests;
- `pandas` to create and clean DataFrames;
- `json` to display JSON more readably;
- `time` to pause between repeated requests;
- `textwrap` only for readable display.

If `requests` is not installed in your environment, install it with:

```python
!pip install requests
```

In [5]:
import requests
import pandas as pd
import json
import time
from pprint import pprint

## 2. A first API request: Open Library Search API

We start with a simple public API. The Open Library search API lets us search for book records.

The endpoint is:

```text
https://openlibrary.org/search.json
```

The parameters specify what we are searching for and how many results we want.

### Key syntax

```python
requests.get(url, params=params, timeout=10)
```

- `url`: API endpoint.
- `params`: dictionary of query parameters. `requests` converts this into a query string.
- `timeout=10`: prevents the notebook from waiting forever if the server does not respond.

Look here for API documentation https://openlibrary.org/dev/docs/api/search

In [2]:
url = "https://openlibrary.org/search.json"

params = {
    "q": "data analytics",
    "limit": 20
}

response = requests.get(url, params=params, timeout=10)

print("Status code:", response.status_code)
print("Final URL sent to server:")
print(response.url)

Status code: 200
Final URL sent to server:
https://openlibrary.org/search.json?q=data+analytics&limit=20


### What to notice

The final URL contains the endpoint plus the query parameters.

A status code of `200` means the server returned a successful response. It does **not** mean that the response contains exactly the records you need, or that the values are ready for analysis.

## 3. Check the response before parsing JSON

A professional notebook should not assume that a request succeeded. Use `raise_for_status()` to stop execution if the server returns an HTTP error.

Common status codes:

| Code | Meaning | Analyst action |
|---:|---|---|
| 200 | Success | Continue, but validate content |
| 400 | Bad request | Check parameters |
| 401/403 | Unauthorized / forbidden | Do not bypass; check access |
| 404 | Not found | Check endpoint or URL |
| 429 | Too many requests | Slow down; respect limits |
| 500 | Server error | Retry later; do not hammer server |

In [3]:
# If the response is not successful, this line raises an error.
response.raise_for_status()

# Convert JSON text into Python objects: dictionaries and lists.
data = response.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['numFound', 'start', 'numFoundExact', 'num_found', 'documentation_url', 'q', 'offset', 'docs'])


## 4. Inspect JSON structure before creating a DataFrame

Do not try this too quickly:

```python
df = pd.DataFrame(response.json())
```

That often creates the wrong table.

First ask:

1. Is the top-level object a dictionary or a list?
2. Where are the actual records?
3. Is there metadata mixed with the data?
4. Are fields nested?

In [6]:
# Pretty-print a few top-level items.
print("Top-level keys:", data.keys())

print("Number of documents found by API:", data.get("numFound"))
print("Number returned in this request:", len(data.get("docs", [])))

print("First record, abbreviated:")
first_doc = data["docs"][0]
for key in list(first_doc.keys())[:15]:
    print(key, ":", first_doc[key])

Top-level keys: dict_keys(['numFound', 'start', 'numFoundExact', 'num_found', 'documentation_url', 'q', 'offset', 'docs'])
Number of documents found by API: 3476
Number returned in this request: 20
First record, abbreviated:
author_key : ['OL769471A', 'OL3503703A']
author_name : ['Foster Provost', 'Tom Fawcett']
cover_edition_key : OL25941419M
cover_i : 7440864
ebook_access : printdisabled
edition_count : 5
first_publish_year : 2013
has_fulltext : True
ia : ['datascienceforbu0000prov']
ia_collection : ['internetarchivebooks', 'printdisabled']
key : /works/OL17363116W
language : ['eng']
public_scan_b : False
title : Data Science for Business


## 5. Convert the records to a DataFrame

For this API, the actual records are stored in:

```python
data["docs"]
```

That is a list of dictionaries, which is convenient for Pandas.

In [7]:
books_raw = pd.DataFrame(data["docs"])

print("Shape:", books_raw.shape)
books_raw.head()

Shape: (20, 16)


,author_key,author_name,cover_edition_key,cover_i,ebook_access,edition_count,first_publish_year,has_fulltext,ia,ia_collection,key,language,public_scan_b,title,lending_edition_s,lending_identifier_s
0,"[OL769471A, OL3503703A]","[Foster Provost, Tom Fawcett]",OL25941419M,7440864.0,printdisabled,5,2013,True,[datascienceforbu0000prov],"[internetarchivebooks, printdisabled]",/works/OL17363116W,[eng],False,Data Science for Business,NaN,NaN
1,[OL11082165A],[Edward Mize],OL46878397M,13619142.0,borrowable,1,2017,True,[dataanalyticsult0000edwa],"[inlibrary, internetarchivebooks, printdisabled]",/works/OL34563643W,NaN,False,Data Analytics,OL46878397M,dataanalyticsult0000edwa
2,"[OL8271295A, OL2680479A]","[Chandan K. Reddy, Charu C. Aggarwal]",OL30390751M,13449796.0,no_ebook,4,2015,False,NaN,NaN,/works/OL22313203W,[eng],False,Healthcare data analytics,NaN,NaN
3,"[OL4097685A, OL7647608A]","[Daniel Memmert, Dominik Raabe]",OL27368732M,8831907.0,no_ebook,8,2018,False,NaN,NaN,/works/OL20183787W,[eng],False,Data Analytics in Football,NaN,NaN
4,[OL7204929A],[EMC Education Services],NaN,NaN,no_ebook,3,2014,False,NaN,NaN,/works/OL21549090W,[eng],False,Data Science and Big Data Analytics,NaN,NaN


## 6. Select the variables we need

API responses often include many fields. We rarely need every column.

Here we select a smaller set of fields for demonstration.

### Why use `.copy()`?

When we create a subset that we plan to modify, `.copy()` avoids confusing chained-assignment behavior in Pandas.

In [10]:
selected_columns = [
    "title",
    "author_name",
    "first_publish_year",
    "language",
    "ebook_access",
    "edition_count",
    "key"
]

# Keep only columns that actually exist in the returned data.
existing_columns = [c for c in selected_columns if c in books_raw.columns]

books = books_raw[existing_columns].copy()

books.head()

,title,author_name,first_publish_year,language,ebook_access,edition_count,key
0,Data Science for Business,"[Foster Provost, Tom Fawcett]",2013,[eng],printdisabled,5,/works/OL17363116W
1,Data Analytics,[Edward Mize],2017,NaN,borrowable,1,/works/OL34563643W
2,Healthcare data analytics,"[Chandan K. Reddy, Charu C. Aggarwal]",2015,[eng],no_ebook,4,/works/OL22313203W
3,Data Analytics in Football,"[Daniel Memmert, Dominik Raabe]",2018,[eng],no_ebook,8,/works/OL20183787W
4,Data Science and Big Data Analytics,[EMC Education Services],2014,[eng],no_ebook,3,/works/OL21549090W


## 7. Understand list-valued fields

Some API fields contain lists. For example, `author_name` may contain multiple authors.

A DataFrame cell can technically contain a list, but that may not be convenient for analysis.

Below we create a simple `first_author` column.

In [11]:
def first_item(value):
    """Return the first item if value is a non-empty list; otherwise return missing."""
    if isinstance(value, list) and len(value) > 0:
        return value[0]
    return pd.NA

books["first_author"] = books["author_name"].apply(first_item)

books[["title", "author_name", "first_author"]].head(10)

,title,author_name,first_author
0,Data Science for Business,"[Foster Provost, Tom Fawcett]",Foster Provost
1,Data Analytics,[Edward Mize],Edward Mize
2,Healthcare data analytics,"[Chandan K. Reddy, Charu C. Aggarwal]",Chandan K. Reddy
3,Data Analytics in Football,"[Daniel Memmert, Dominik Raabe]",Daniel Memmert
4,Data Science and Big Data Analytics,[EMC Education Services],EMC Education Services
5,Big Data Analytics,"[Parag Kulkarni, Sarang Joshi, Meta S. Brown]",Parag Kulkarni
6,Data Analytics for Accounting,"[Vernon J. Richardson, Ryan Teeter, Katie Terr...",Vernon J. Richardson
7,Big Data Analytics,[Frank J. Ohlhorst],Frank J. Ohlhorst
8,Data Analytics,"[Shuai Huang, Houtao Deng]",Shuai Huang
9,Data Analytics and Decision Making,[Ali AbdulHussein],Ali AbdulHussein


## 8. Convert data types deliberately

Even when an API looks structured, you should inspect data types.

In [12]:
books.dtypes

title                 object
author_name           object
first_publish_year     int64
language              object
ebook_access          object
edition_count          int64
key                   object
first_author          object
dtype: object

`first_publish_year` and `edition_count` should be numeric for analysis.

`pd.to_numeric(..., errors="coerce")` means:

- convert valid values to numbers;
- if a value cannot be converted, turn it into `NaN` instead of crashing the notebook.

In [13]:
if "first_publish_year" in books.columns:
    books["first_publish_year"] = pd.to_numeric(
        books["first_publish_year"],
        errors="coerce"
    )

if "edition_count" in books.columns:
    books["edition_count"] = pd.to_numeric(
        books["edition_count"],
        errors="coerce"
    )

books.dtypes

title                 object
author_name           object
first_publish_year     int64
language              object
ebook_access          object
edition_count          int64
key                   object
first_author          object
dtype: object

## 9. Basic validation checks

Do not stop after creating a DataFrame. Validate what you collected.

In [14]:
validation_summary = pd.Series({
    "rows_returned": len(books),
    "missing_title": books["title"].isna().sum() if "title" in books else pd.NA,
    "missing_first_author": books["first_author"].isna().sum() if "first_author" in books else pd.NA,
    "missing_first_publish_year": books["first_publish_year"].isna().sum() if "first_publish_year" in books else pd.NA,
    "duplicate_keys": books["key"].duplicated().sum() if "key" in books else pd.NA
})

validation_summary

rows_returned                 20
missing_title                  0
missing_first_author           0
missing_first_publish_year     0
duplicate_keys                 0
dtype: int64

## 10. A small analysis after validation

This is not a full EDA exercise. We only check whether the API-derived fields are usable.

In [15]:
if "first_publish_year" in books.columns:
    display(
        books[["title", "first_author", "first_publish_year", "edition_count"]]
        .sort_values("first_publish_year")
        .head(10)
    )

,title,first_author,first_publish_year,edition_count
7,Big Data Analytics,Frank J. Ohlhorst,2012,5
13,Data Analytics,Thomas A. Runkler,2012,6
0,Data Science for Business,Foster Provost,2013,5
4,Data Science and Big Data Analytics,EMC Education Services,2014,3
18,Big data analytics for beginners,Faraz Rabbani,2014,6
14,Fundamentals of Machine Learning for Predictiv...,John D. Kelleher,2015,4
2,Healthcare data analytics,Chandan K. Reddy,2015,4
5,Big Data Analytics,Parag Kulkarni,2016,1
15,Data Analytics for Beginners,Anthony Williams,2017,2
17,Data Analytics For Beginners,Tech World,2017,1


# Part 2 — Repeated API Requests and Issue Logs

Now we request several search terms. In a real project, some requests may fail or return unexpected structures.

A good notebook records these issues instead of silently ignoring them.

In [16]:
search_terms = [
    "data analytics",
    "business analytics",
    "data visualization"
]

all_books = []
issues = []

for term in search_terms:
    params = {
        "q": term,
        "limit": 10
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        result = r.json()

        docs = result.get("docs", [])

        if len(docs) == 0:
            issues.append({
                "search_term": term,
                "url": r.url,
                "issue": "No documents returned"
            })

        for doc in docs:
            all_books.append({
                "search_term": term,
                "title": doc.get("title"),
                "first_author": first_item(doc.get("author_name")),
                "first_publish_year": doc.get("first_publish_year"),
                "edition_count": doc.get("edition_count"),
                "key": doc.get("key"),
                "source_url": r.url
            })

    except Exception as e:
        issues.append({
            "search_term": term,
            "url": url,
            "issue": str(e)
        })

    # Pause briefly between requests.
    time.sleep(0.5)

books_multi = pd.DataFrame(all_books)
issue_log = pd.DataFrame(issues)

print("Rows collected:", len(books_multi))
print("Issues logged:", len(issue_log))
books_multi.head()

Rows collected: 20
Issues logged: 1


,search_term,title,first_author,first_publish_year,edition_count,key,source_url
0,data analytics,Data Science for Business,Foster Provost,2013,5,/works/OL17363116W,https://openlibrary.org/search.json?q=data+ana...
1,data analytics,Data Analytics,Edward Mize,2017,1,/works/OL34563643W,https://openlibrary.org/search.json?q=data+ana...
2,data analytics,Healthcare data analytics,Chandan K. Reddy,2015,4,/works/OL22313203W,https://openlibrary.org/search.json?q=data+ana...
3,data analytics,Data Analytics in Football,Daniel Memmert,2018,8,/works/OL20183787W,https://openlibrary.org/search.json?q=data+ana...
4,data analytics,Data Science and Big Data Analytics,EMC Education Services,2014,3,/works/OL21549090W,https://openlibrary.org/search.json?q=data+ana...


## 11. Validate repeated requests

Check whether pagination, repeated terms, or the API response created duplicate records.

In [18]:
books_multi["first_publish_year"] = pd.to_numeric(
    books_multi["first_publish_year"],
    errors="coerce"
)

books_multi["edition_count"] = pd.to_numeric(
    books_multi["edition_count"],
    errors="coerce"
)

print("Shape:", books_multi.shape)
print("Missing values:")
print(books_multi.isna().sum())

print("Duplicate book keys:")
print(books_multi.duplicated(subset=["key"]).sum())

print("Rows by search term:")
print(books_multi["search_term"].value_counts())

issue_log

Shape: (20, 7)
Missing values:
search_term           0
title                 0
first_author          0
first_publish_year    0
edition_count         0
key                   0
source_url            0
dtype: int64
Duplicate book keys:
0
Rows by search term:
search_term
data analytics        10
business analytics    10
Name: count, dtype: int64


,search_term,url,issue
0,data visualization,https://openlibrary.org/search.json,"HTTPSConnectionPool(host='openlibrary.org', po..."


### Why might duplicate keys occur?

The same book may appear under multiple search terms. That is not necessarily a data error, but it matters for analysis.

If the question is "How many search results did we collect?" keep duplicates by search term.

If the question is "How many unique books did we collect?" deduplicate by `key`.

In [19]:
unique_books = books_multi.drop_duplicates(subset=["key"]).copy()

print("Search-result rows:", len(books_multi))
print("Unique books:", len(unique_books))

Search-result rows: 20
Unique books: 20


# Part 3 — Census API Example

First register for an API Key here https://www.census.gov/data/developers/data-sets.html (click on Request a Key)
The Census API has a different response shape. Many Census endpoints return an **array of arrays**:

```python
[
    [header1, header2, ...],
    [row1_value1, row1_value2, ...],
    [row2_value1, row2_value2, ...]
]
```

The first row contains column names. The remaining rows contain data.

This example uses California places and two profile variables:

- total population estimate;
- median household income estimate.

If the endpoint year changes in the future, update the year in the URL.

In [ ]:
census_url = "https://api.census.gov/data/2023/acs/acs5/profile"

census_params = {
    "get": "NAME,DP05_0001E,DP03_0062E",
    "for": "place:*",
    "in": "state:06",
    "key": "YOUR_KEY"
}

census_response = requests.get(
    census_url,
    params=census_params,
    timeout=20
)

print("Status code:", census_response.status_code)
print("Final URL:")
print(census_response.url)

census_response.raise_for_status()

In [7]:
print("Status code:", census_response.status_code)
print("Content type:", census_response.headers.get("content-type"))
print("Response text:")
print(census_response.text[:500])

Status code: 200
Content type: application/json;charset=utf-8
Response text:
[["NAME","DP05_0001E","DP03_0062E","state","place"],
["Acalanes Ridge CDP, California","1025","172667","06","00135"],
["Acampo CDP, California","278","-666666666","06","00156"],
["Acton CDP, California","7721","110000","06","00212"],
["Adelanto city, California","38010","68419","06","00296"],
["Adin CDP, California","91","36250","06","00310"],
["Agoura Hills city, California","19881","171944","06","00394"],
["Agua Dulce CDP, California","2768","124268","06","00450"],
["Aguanga CDP, California","


In [8]:
print(census_response.status_code)
print(census_response.url)
print(census_response.text[:1000])

200
https://api.census.gov/data/2023/acs/acs5/profile?get=NAME%2CDP05_0001E%2CDP03_0062E&for=place%3A%2A&in=state%3A06&key=b97fbf2757be734b8ed0cdc155e85d9b89837063
[["NAME","DP05_0001E","DP03_0062E","state","place"],
["Acalanes Ridge CDP, California","1025","172667","06","00135"],
["Acampo CDP, California","278","-666666666","06","00156"],
["Acton CDP, California","7721","110000","06","00212"],
["Adelanto city, California","38010","68419","06","00296"],
["Adin CDP, California","91","36250","06","00310"],
["Agoura Hills city, California","19881","171944","06","00394"],
["Agua Dulce CDP, California","2768","124268","06","00450"],
["Aguanga CDP, California","756","-666666666","06","00464"],
["Ahwahnee CDP, California","1532","65980","06","00478"],
["Airport CDP, California","1258","42000","06","00535"],
["Alameda city, California","76876","132015","06","00562"],
["Alamo CDP, California","14441","250001","06","00618"],
["Albany city, California","19768","135927","06","00674"],
["Albion CDP

## 12. Inspect the Census JSON structure

In [9]:
census_data = census_response.json()

print(type(census_data))
print("Number of rows including header:", len(census_data))
print("Header row:")
print(census_data[0])
print("First data row:")
print(census_data[1])

<class 'list'>
Number of rows including header: 1619
Header row:
['NAME', 'DP05_0001E', 'DP03_0062E', 'state', 'place']
First data row:
['Acalanes Ridge CDP, California', '1025', '172667', '06', '00135']


## 13. Convert array-of-arrays to a DataFrame

Because the first row is the header:

```python
pd.DataFrame(census_data[1:], columns=census_data[0])
```

In [10]:
places_raw = pd.DataFrame(
    census_data[1:],
    columns=census_data[0]
)

print(places_raw.shape)
places_raw.head()

(1618, 5)


,NAME,DP05_0001E,DP03_0062E,state,place
0,"Acalanes Ridge CDP, California",1025,172667,06,00135
1,"Acampo CDP, California",278,-666666666,06,00156
2,"Acton CDP, California",7721,110000,06,00212
3,"Adelanto city, California",38010,68419,06,00296
4,"Adin CDP, California",91,36250,06,00310


## 14. Rename columns and convert types

The API variable names are meaningful only if you know the Census documentation.

For analysis, rename them to readable names while documenting the original codes.

In [11]:
places = places_raw.rename(columns={
    "NAME": "place_name",
    "DP05_0001E": "population",
    "DP03_0062E": "median_household_income",
    "state": "state_code",
    "place": "place_code"
}).copy()

places["population"] = pd.to_numeric(
    places["population"],
    errors="coerce"
)

places["median_household_income"] = pd.to_numeric(
    places["median_household_income"],
    errors="coerce"
)

# State and place codes are identifiers, not measurements.
places["state_code"] = places["state_code"].astype("string")
places["place_code"] = places["place_code"].astype("string")

places.dtypes

place_name                         object
population                          int64
median_household_income             int64
state_code                 string[python]
place_code                 string[python]
dtype: object

## 15. Validate the Census-derived table

In [12]:
places_validation = pd.Series({
    "rows": len(places),
    "unique_place_codes": places["place_code"].nunique(),
    "duplicate_place_codes": places["place_code"].duplicated().sum(),
    "missing_population": places["population"].isna().sum(),
    "missing_income": places["median_household_income"].isna().sum(),
    "nonpositive_population": (places["population"] <= 0).sum(),
    "negative_income": (places["median_household_income"] < 0).sum()
})

places_validation

rows                      1618
unique_place_codes        1618
duplicate_place_codes        0
missing_population           0
missing_income               0
nonpositive_population      41
negative_income            239
dtype: int64

## 16. Create an analysis subset

Suppose management is interested in larger places only.

The filter below is a business/analytical decision. Document how many records remain.

In [13]:
large_places = places[places["population"] >= 50000].copy()

print("Original number of places:", len(places))
print("Places with population >= 50,000:", len(large_places))

large_places[["place_name", "population", "median_household_income"]].head()

Original number of places: 1618
Places with population >= 50,000: 188


,place_name,population,median_household_income
10,"Alameda city, California",76876,132015
15,"Alhambra city, California",81483,85189
17,"Aliso Viejo city, California",51320,137970
37,"Anaheim city, California",344553,90583
43,"Antioch city, California",115759,94256


## 17. Simple ranking for discussion

This is not an optimization model or a final site-selection recommendation. It is a quick descriptive summary from API data.

In [14]:
large_places_sorted = (
    large_places
    .sort_values(["median_household_income", "population"], ascending=False)
    [["place_name", "population", "median_household_income"]]
)

large_places_sorted.head(10)

,place_name,population,median_household_income
343,"Cupertino city, California",58886,231139
1077,"Palo Alto city, California",67231,220408
402,"Dublin city, California",70542,205046
1292,"San Ramon city, California",85734,197358
1138,"Pleasanton city, California",77500,186206
1414,"Sunnyvale city, California",153455,181862
978,"Mountain View city, California",82363,179917
929,"Milpitas city, California",78216,176822
529,"Fremont city, California",228518,176350
1296,"Santa Clara city, California",129239,173670


# Part 4 — Saving API-Derived Outputs

A reproducible API workflow should save:

1. the cleaned dataset;
2. a validation summary;
3. an issue log if relevant.

In [ ]:
books_multi.to_csv("week4_openlibrary_books.csv", index=False)
issue_log.to_csv("week4_openlibrary_issue_log.csv", index=False)
large_places.to_csv("week4_census_large_ca_places.csv", index=False)

print("Saved files:")
print("- week4_openlibrary_books.csv")
print("- week4_openlibrary_issue_log.csv")
print("- week4_census_large_ca_places.csv")

# Practice Exercise — API Market Data Mini-Case

## Business scenario

A specialty retailer is considering expansion in California. Management wants a preliminary list of places with:

- population of at least 75,000;
- available median household income data;
- relatively high income compared with other places in the filtered dataset.

Use the Census API example above.

## Your tasks

1. Re-run the Census API request.
2. Convert the response to a DataFrame.
3. Rename columns.
4. Convert population and income to numeric values.
5. Filter to places with population at least 75,000.
6. Remove rows with missing income only for this specific ranking exercise.
7. Create a table of the top 10 places by median household income.
8. Create a validation summary.
9. Write 3–5 sentences explaining whether this dataset alone is enough for a real expansion decision.

### Important

Do not claim that the top 10 places are necessarily the best expansion locations. This API gives only a few market indicators. A real decision would need competitor data, rent, traffic, store availability, demographics, and operating constraints.

## Practice reflection prompt

Answer in 3–5 sentences:

1. What can this API-derived dataset tell management?
2. What can it **not** tell management?
3. What additional data would you want before making a real expansion recommendation?
4. Which validation checks were most important?

# End of Week 4 Notebook

Key lesson:

> APIs can make data acquisition more structured and reproducible, but the analyst is still responsible for inspecting, cleaning, validating, documenting, and interpreting the data.